# RadOptLIB quick start

This notebook walks through the `radoptlib` Python package: finding instances, downloading and loading
them, evaluating a solution, solving an instance with a solver, and writing an MPS file.

It uses the smallest instance of the library (`rt-qp-0062`, a 15 MB file) and runs in about a minute.
The same steps work for every instance; the largest ones need a machine with a large memory.

## 1. Install

```bash
pip install git+https://github.com/Radiotherapy-Optimization/RadOptLIB.git
```

The package needs only `numpy`, `scipy` and `h5py`. It can also be used from a clone of the repository
without installing it, by adding the clone's `src` folder to `PYTHONPATH`.

Section 6 of this notebook additionally uses the open-source solver Clarabel (`pip install clarabel`).

## 2. Find instances

Every problem class has a manifest, a table with one row per instance: its name, the disease site, the
type, the problem size, the file size and the download addresses. `list_instances` returns the rows and
`info` the row of one instance. The columns are described in
[docs/qp.md](../docs/qp.md#manifest-columns-manifestqpcsv).

In [1]:
import radoptlib

rows = radoptlib.list_instances('qp')
print(len(rows), 'instances')
print('name        site       type  size    n_vars  n_ineq_cons   file size')
for r in rows[:5]:
    print(f"{r['name']}  {r['disease_site']:10s} {r['type']:4d}  {r['size']:7s} {r['n_vars']:6d}  {r['n_ineq_cons']:11d}  {r['file_size_bytes'] / 1e6:6.0f} MB")

142 instances
name        site       type  size    n_vars  n_ineq_cons   file size
rt-qp-0001  lung          1  medium   26585        83761      80 MB
rt-qp-0002  lung          2  medium   99444       172908     129 MB
rt-qp-0003  lung          0  medium   87773       122332      94 MB
rt-qp-0004  lung          1  small    16477        72395     120 MB
rt-qp-0005  lung          2  medium  116338       147856      74 MB


In [2]:
radoptlib.info('rt-qp-0062')

{'name': 'rt-qp-0062',
 'problem_class': 'qp',
 'problem_type': 'convex continuous QP',
 'patient': 'oligomets-0012',
 'size': 'small',
 'n_vars': 8665,
 'n_ineq_cons': 27313,
 'n_eq_cons': 0,
 'nnz_Q': 1238488,
 'nnz_G': 1289937,
 'file_size_bytes': 14731571,
 'disease_site': 'oligomets',
 'modality': 'vmat',
 'protocol': 'oligomets_10gy_3fx',
 'type': 0,
 'url': '',
 'md5': '',
 'sol_url': '',
 'sol_md5': '',
 'added_in_version': '1.0',
 'note': ''}

`select` returns the names of the instances that match a filter on any manifest column. A list means
"any of", and numeric columns also accept `min_` and `max_` limits.

In [3]:
print(radoptlib.select(disease_site='lung', type=1))
print(len(radoptlib.select(size=['small', 'medium'])), 'small or medium instances')

names = radoptlib.select(size='small', max_file_size_bytes=20e6)
print(names)

['rt-qp-0001', 'rt-qp-0004', 'rt-qp-0007', 'rt-qp-0010', 'rt-qp-0013', 'rt-qp-0016', 'rt-qp-0019', 'rt-qp-0021', 'rt-qp-0023', 'rt-qp-0025', 'rt-qp-0027']
122 small or medium instances
['rt-qp-0062', 'rt-qp-0064']


## 3. Download and load

`download` takes one name, a list of names, or a problem class with filters. Files are stored in
`~/.radoptlib` (or the folder given in the `RADOPTLIB_DATA` environment variable). Every download is
checked against the checksum in the manifest, and files that are already there are not downloaded again.

In [4]:
paths = radoptlib.download(names)
print([path.name for path in paths])

2 instance(s), 0 to download (0.00 GB)
['rt-qp-0062.h5', 'rt-qp-0064.h5']


`load` reads an instance into a `Problem`: the QP

```
minimize    ½ x'Qx + q'x + c
subject to  Gx ≤ h,  l ≤ x ≤ u
```

with sparse matrices `Q` and `G`, vectors `q`, `h`, `l`, `u` and a scalar `c`. A missing bound is `-inf`
in `l` or `inf` in `u`.

In [5]:
import numpy as np

p = radoptlib.load('rt-qp-0062')
print(p)
print('Q:', p.Q.shape, ' G:', p.G.shape)
print('variables with a lower bound:', int(np.isfinite(p.l).sum()), ' with an upper bound:', int(np.isfinite(p.u).sum()))
p.info

1 instance(s), 0 to download (0.00 GB)
<Problem rt-qp-0062: n=8665 m=27313 nnz_Q=1238488 nnz_G=1289937>
Q: (8665, 8665)  G: (27313, 8665)
variables with a lower bound: 8665  with an upper bound: 4016


{'name': 'rt-qp-0062',
 'problem_class': 'qp',
 'problem_type': 'convex continuous QP',
 'disease_site': 'oligomets',
 'modality': 'vmat',
 'type': 0,
 'patient': 'oligomets-0012',
 'protocol': 'oligomets_10gy_3fx',
 'library_version': '1.0',
 'format_version': '1.0',
 'license': 'CC BY 4.0',
 'repo_url': 'https://github.com/Radiotherapy-Optimization/RadOptLIB'}

## 4. Evaluate the reference solution

Every instance comes with a reference solution computed with
[Warm-IP](https://github.com/Radiotherapy-Optimization/Warm-IP): the primal vector `x` and the
multipliers of `Gx ≤ h`, `x ≥ l` and `x ≤ u`. `load_solution` downloads it on first use.

In [6]:
sol = radoptlib.load_solution('rt-qp-0062')
for key, value in sol.items():
    print(f'{key:17s}', value.shape if hasattr(value, 'shape') else value)

name              rt-qp-0062
solver            Warm-IP, https://github.com/Radiotherapy-Optimization/Warm-IP
format_note       Reference solution for the QP of the same name: minimize 0.5*x'Qx + q'x + c subject to Gx <= h, l <= x <= u. x is the primal solution; dual_ineq_cons, dual_lower_bound and dual_upper_bound are the nonnegative multipliers of Gx <= h, x >= l and x <= u, so that Qx + q + G'dual_ineq_cons - dual_lower_bound + dual_upper_bound = 0.
x                 (8665,)
dual_ineq_cons    (27313,)
dual_lower_bound  (8665,)
dual_upper_bound  (8665,)


`evaluate_solution` returns the objective value and the KKT residuals of a solution: the primal residual
(constraint violation), the dual residual (stationarity), the duality gap and the dual infeasibility.
All of them are relative; the formulas are in [docs/qp.md](../docs/qp.md#evaluating-a-solution).

In [7]:
def show(result):
    for key, value in result.items():
        print(f'{key:12s}', f'{value:.6f}' if key == 'objective' else f'{value:.1e}')

reference = radoptlib.evaluate_solution(p, sol['x'], sol['dual_ineq_cons'], sol['dual_lower_bound'],
                                        sol['dual_upper_bound'])
show(reference)

objective    10428.874667
primal_res   7.0e-17
dual_res     4.9e-14
gap          6.2e-09
dual_infeas  0.0e+00


## 5. Write an MPS file

For solvers that read MPS files, `to_mps` writes the instance as a free-format MPS file with a `QUADOBJ`
section (gzipped when the name ends in `.gz`). Users without Python can do the same with the stand-alone
program `radoptlib2mps`, described in [mps/README.md](../mps/README.md).

In [8]:
import os

mps_path = radoptlib.to_mps(p, 'rt-qp-0062.mps.gz')
print(mps_path, f'{os.path.getsize(mps_path) / 1e6:.0f} MB')

rt-qp-0062.mps.gz 21 MB


## 6. Solve the instance with a solver

Any QP solver can be used. This section uses the open-source solver
[Clarabel](https://github.com/oxfordcontrol/Clarabel.rs) as an example; it is not part of RadOptLIB and
is not needed for anything else.

Clarabel solves `minimize ½ x'Px + q'x subject to Ax + s = b, s ≥ 0`, so the bounds are passed as
constraints. The three groups `Gx ≤ h`, `x ≥ l` and `x ≤ u` are stacked into one matrix, using only the
finite bounds.

In [9]:
import clarabel
import scipy.sparse as sp

n, m = p.n, p.m
has_l, has_u = np.isfinite(p.l), np.isfinite(p.u)
I = sp.eye(n, format='csc')

A = sp.vstack([p.G, -I[has_l], I[has_u]], format='csc')        # Gx <= h,  -x <= -l,  x <= u
b = np.concatenate([p.h, -p.l[has_l], p.u[has_u]])

settings = clarabel.DefaultSettings()
settings.verbose = False
solver = clarabel.DefaultSolver(sp.triu(p.Q, format='csc'), p.q, A, b,
                                [clarabel.NonnegativeConeT(len(b))], settings)
result = solver.solve()
print(result.status)

Solved


The solver returns one multiplier per row of `A`. They are split back into the three vectors that
`evaluate_solution` expects; a variable without a bound gets a zero multiplier.

In [10]:
x = np.array(result.x)
z = np.array(result.z)

dual_ineq_cons = z[:m]
dual_lower_bound = np.zeros(n)
dual_lower_bound[has_l] = z[m:m + has_l.sum()]
dual_upper_bound = np.zeros(n)
dual_upper_bound[has_u] = z[m + has_l.sum():]

mine = radoptlib.evaluate_solution(p, x, dual_ineq_cons, dual_lower_bound, dual_upper_bound)
show(mine)

difference = abs(mine['objective'] - reference['objective']) / abs(reference['objective'])
print(f'relative difference from the reference objective: {difference:.1e}')

objective    10428.874645
primal_res   2.4e-17
dual_res     2.8e-13
gap          2.2e-09
dual_infeas  0.0e+00
relative difference from the reference objective: 2.1e-09


## Functions of the package

| Function | Purpose |
|---|---|
| `list_instances(problem_class)`, `info(name)` | manifest rows of a class, or of one instance |
| `select(**filters)` | names of the instances that match filters on manifest columns |
| `download(name_or_class, ...)` | download instances |
| `load(name_or_path)` | read an instance into a `Problem` |
| `load_solution(name_or_path, solver='warmip')` | read a reference solution |
| `evaluate_solution(problem, x, dual_ineq_cons=None, dual_lower_bound=None, dual_upper_bound=None, tol=None)` | objective value and KKT residuals of a solution |
| `to_mps(problem, path)` | write an instance as an MPS file |

The files can also be read without the package, in Python or in other languages; the layout is described
in [docs/qp.md](../docs/qp.md#instance-files).